# 13. Instacart Cold Start 모델 Pickle 생성

> **목표**: `instacart_cold_start_v1.pkl` 생성

## Pickle 구조
```python
instacart_model = {
    'version': '1.0.0',
    'created_at': datetime,
    'metadata': {...},
    'components': {
        'time_patterns': {...},
        'category_mapping': {...},
        'aisle_stats': {...},
        'global_popular_aisles': [...],
    },
    'hyperparameters': {...},
}
```

In [1]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
from datetime import datetime
import json

DATA_DIR = Path('../data/instacart')
PROCESSED_DIR = Path('../data/processed/instacart')
MODEL_DIR = Path('../pred/models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

## 1. 이전 단계 데이터 로드

In [2]:
# 시간 패턴 로드
with open(PROCESSED_DIR / 'time_patterns.pkl', 'rb') as f:
    time_patterns = pickle.load(f)
print(f"시간 패턴 로드 완료: {len(time_patterns)}개")

# 카테고리 매핑 로드
with open(PROCESSED_DIR / 'category_mapping.pkl', 'rb') as f:
    category_mapping = pickle.load(f)
print(f"카테고리 매핑 로드 완료: {len(category_mapping)}개")

# Aisle 통계 로드
aisle_stats_df = pd.read_csv(PROCESSED_DIR / 'aisle_stats.csv')
print(f"Aisle 통계 로드 완료: {len(aisle_stats_df)}개")

시간 패턴 로드 완료: 168개
카테고리 매핑 로드 완료: 89개
Aisle 통계 로드 완료: 134개


In [3]:
# 원본 데이터에서 추가 통계 계산
orders = pd.read_csv(DATA_DIR / 'orders.csv')
order_products = pd.read_csv(DATA_DIR / 'order_products__prior.csv')

total_orders = len(orders)
total_users = orders['user_id'].nunique()
total_products_ordered = len(order_products)

print(f"\n=== 데이터 통계 ===")
print(f"총 주문 수: {total_orders:,}")
print(f"총 유저 수: {total_users:,}")
print(f"총 주문-상품 레코드: {total_products_ordered:,}")


=== 데이터 통계 ===
총 주문 수: 3,421,083
총 유저 수: 206,209
총 주문-상품 레코드: 32,434,489


## 2. Aisle 통계 구조화

In [4]:
# Aisle 통계를 딕셔너리로 변환
aisle_stats = {}
for _, row in aisle_stats_df.iterrows():
    aisle_stats[int(row['aisle_id'])] = {
        'aisle_name': row['aisle'],
        'department': row['department'],
        'order_count': int(row['order_count']),
        'reorder_rate': float(row['reorder_rate']),
        'avg_cart_position': float(row['avg_cart_position']),
    }

print(f"Aisle 통계 구조화 완료: {len(aisle_stats)}개")

Aisle 통계 구조화 완료: 134개


In [5]:
# 글로벌 인기 Aisle (시간 무관, 전체 주문 기준 Top 20)
global_popular_aisles = aisle_stats_df.sort_values('order_count', ascending=False).head(20)['aisle_id'].tolist()
global_popular_aisles = [int(x) for x in global_popular_aisles]

print(f"\n글로벌 인기 Aisle Top 20:")
for rank, aisle_id in enumerate(global_popular_aisles, 1):
    aisle_info = aisle_stats[aisle_id]
    print(f"  {rank}. [{aisle_id}] {aisle_info['aisle_name']}: {aisle_info['order_count']:,}건")


글로벌 인기 Aisle Top 20:
  1. [24] fresh fruits: 1,790,771건
  2. [83] fresh vegetables: 1,427,631건
  3. [123] packaged vegetables fruits: 1,179,243건
  4. [120] yogurt: 847,081건
  5. [84] milk: 785,987건
  6. [21] packaged cheese: 737,899건
  7. [115] water seltzer sparkling water: 614,081건
  8. [91] soy lactosefree: 545,714건
  9. [107] chips pretzels: 538,052건
  10. [112] bread: 527,129건
  11. [86] eggs: 440,410건
  12. [31] refrigerated: 429,510건
  13. [116] frozen produce: 395,743건
  14. [78] crackers: 368,577건
  15. [37] ice cream ice: 352,768건
  16. [96] lunch meat: 334,151건
  17. [67] fresh dips tapenades: 314,910건
  18. [16] fresh herbs: 300,364건
  19. [121] cereal: 297,307건
  20. [53] cream: 295,032건


## 3. 최종 Pickle 구조 생성

In [6]:
# 최종 Pickle 구조
instacart_model = {
    'version': '1.0.0',
    'created_at': datetime.now().isoformat(),
    'model_type': 'instacart_cold_start',
    
    'metadata': {
        'total_orders': total_orders,
        'total_users': total_users,
        'total_products_ordered': total_products_ordered,
        'time_patterns_count': len(time_patterns),
        'mapped_aisles': len(category_mapping),
        'total_aisles': len(aisle_stats),
        'data_source': 'Instacart Online Grocery Shopping Dataset',
        'description': '32M Instacart 주문 데이터 기반 Cold Start 추천 모델',
    },
    
    'components': {
        # 168개 시간 패턴 (24시간 × 7요일)
        'time_patterns': time_patterns,
        
        # Instacart aisle → SelF category 매핑
        'category_mapping': category_mapping,
        
        # Aisle별 통계 (재주문율, 평균 장바구니 위치 등)
        'aisle_stats': aisle_stats,
        
        # 글로벌 인기 Aisle (시간 무관)
        'global_popular_aisles': global_popular_aisles,
    },
    
    'hyperparameters': {
        'top_aisles_per_pattern': 10,
        'min_orders_threshold': 100,
        'time_context_ranges': {
            'morning': (6, 11),
            'lunch': (11, 14),
            'afternoon': (14, 17),
            'dinner': (17, 21),
            'night': (21, 6),
        },
    },
}

print("Pickle 구조 생성 완료")
print(f"\n=== 구조 요약 ===")
print(f"version: {instacart_model['version']}")
print(f"created_at: {instacart_model['created_at']}")
print(f"metadata keys: {list(instacart_model['metadata'].keys())}")
print(f"components keys: {list(instacart_model['components'].keys())}")
print(f"hyperparameters keys: {list(instacart_model['hyperparameters'].keys())}")

Pickle 구조 생성 완료

=== 구조 요약 ===
version: 1.0.0
created_at: 2025-12-14T22:18:15.634758
metadata keys: ['total_orders', 'total_users', 'total_products_ordered', 'time_patterns_count', 'mapped_aisles', 'total_aisles', 'data_source', 'description']
components keys: ['time_patterns', 'category_mapping', 'aisle_stats', 'global_popular_aisles']
hyperparameters keys: ['top_aisles_per_pattern', 'min_orders_threshold', 'time_context_ranges']


## 4. Pickle 저장

In [7]:
# Pickle 저장
pickle_path = MODEL_DIR / 'instacart_cold_start_v1.pkl'

with open(pickle_path, 'wb') as f:
    pickle.dump(instacart_model, f)

file_size = pickle_path.stat().st_size / 1024 / 1024
print(f"\nPickle 저장 완료: {pickle_path}")
print(f"파일 크기: {file_size:.2f} MB")

# 목표: 50MB 이하
print(f"크기 제한 (< 50MB): {'✓ PASS' if file_size < 50 else '✗ FAIL'}")


Pickle 저장 완료: ..\pred\models\instacart_cold_start_v1.pkl
파일 크기: 0.08 MB
크기 제한 (< 50MB): ✓ PASS


## 5. 로드 테스트

In [8]:
import time

# 로드 시간 측정
start_time = time.time()
with open(pickle_path, 'rb') as f:
    loaded_model = pickle.load(f)
load_time = time.time() - start_time

print(f"로드 시간: {load_time:.3f}초")
print(f"로드 시간 제한 (< 1초): {'✓ PASS' if load_time < 1 else '✗ FAIL'}")

로드 시간: 0.002초
로드 시간 제한 (< 1초): ✓ PASS


In [9]:
# 모든 컴포넌트 접근 테스트
print("\n=== 컴포넌트 접근 테스트 ===")

# 1. 버전 확인
print(f"✓ version: {loaded_model['version']}")

# 2. 시간 패턴 접근
sample_pattern = loaded_model['components']['time_patterns'][(1, 10)]  # 월요일 오전 10시
print(f"✓ time_patterns[(1, 10)]: {sample_pattern['total_orders']:,} orders")

# 3. 카테고리 매핑 접근
sample_mapping = loaded_model['components']['category_mapping'].get(24)  # fresh fruits
print(f"✓ category_mapping[24]: {sample_mapping}")

# 4. Aisle 통계 접근
sample_aisle = loaded_model['components']['aisle_stats'][24]
print(f"✓ aisle_stats[24]: {sample_aisle['aisle_name']} (재주문율: {sample_aisle['reorder_rate']:.1%})")

# 5. 글로벌 인기 Aisle
top_aisle = loaded_model['components']['global_popular_aisles'][0]
print(f"✓ global_popular_aisles[0]: {top_aisle}")

print("\n모든 컴포넌트 접근 테스트 통과!")


=== 컴포넌트 접근 테스트 ===
✓ version: 1.0.0
✓ time_patterns[(1, 10)]: 52,999 orders
✓ category_mapping[24]: 1
✓ aisle_stats[24]: fresh fruits (재주문율: 71.8%)
✓ global_popular_aisles[0]: 24

모든 컴포넌트 접근 테스트 통과!


## 6. 추천 시뮬레이션 테스트

In [10]:
def simulate_cold_start_recommendation(
    model,
    day_of_week: int,
    hour_of_day: int,
    limit: int = 5,
):
    """Cold Start 추천 시뮬레이션"""
    # 시간 패턴에서 인기 Aisle 조회
    pattern = model['components']['time_patterns'].get((day_of_week, hour_of_day))
    
    if not pattern:
        print(f"패턴 없음: ({day_of_week}, {hour_of_day})")
        return []
    
    # Top Aisle에서 SelF 카테고리로 매핑
    category_mapping = model['components']['category_mapping']
    
    recommendations = []
    for aisle in pattern['top_aisles'][:limit]:
        aisle_id = aisle['aisle_id']
        self_category = category_mapping.get(aisle_id)
        
        recommendations.append({
            'aisle_id': aisle_id,
            'aisle_name': aisle['aisle_name'],
            'self_category_id': self_category,
            'order_count': aisle['order_count'],
            'reorder_rate': aisle['reorder_rate'],
        })
    
    return recommendations

# 테스트: 월요일 오전 8시
print("=== Cold Start 추천 시뮬레이션: 월요일 오전 8시 ===")
recommendations = simulate_cold_start_recommendation(loaded_model, 1, 8, limit=5)
for rec in recommendations:
    cat_str = f"→ SelF 카테고리 {rec['self_category_id']}" if rec['self_category_id'] else "(매핑 없음)"
    print(f"  {rec['aisle_name']}: {rec['order_count']:,}건, 재주문율 {rec['reorder_rate']:.1%} {cat_str}")

print("\n=== Cold Start 추천 시뮬레이션: 토요일 저녁 7시 ===")
recommendations = simulate_cold_start_recommendation(loaded_model, 6, 19, limit=5)
for rec in recommendations:
    cat_str = f"→ SelF 카테고리 {rec['self_category_id']}" if rec['self_category_id'] else "(매핑 없음)"
    print(f"  {rec['aisle_name']}: {rec['order_count']:,}건, 재주문율 {rec['reorder_rate']:.1%} {cat_str}")

=== Cold Start 추천 시뮬레이션: 월요일 오전 8시 ===
  fresh fruits: 20,192건, 재주문율 77.0% → SelF 카테고리 1
  fresh vegetables: 12,865건, 재주문율 63.6% → SelF 카테고리 2
  packaged vegetables fruits: 12,002건, 재주문율 68.7% → SelF 카테고리 1
  milk: 9,574건, 재주문율 82.7% → SelF 카테고리 20
  yogurt: 9,475건, 재주문율 74.3% → SelF 카테고리 21

=== Cold Start 추천 시뮬레이션: 토요일 저녁 7시 ===
  fresh fruits: 9,357건, 재주문율 69.5% → SelF 카테고리 1
  fresh vegetables: 7,807건, 재주문율 58.8% → SelF 카테고리 2
  packaged vegetables fruits: 6,238건, 재주문율 61.7% → SelF 카테고리 1
  yogurt: 4,572건, 재주문율 65.3% → SelF 카테고리 21
  milk: 4,262건, 재주문율 76.9% → SelF 카테고리 20


## 7. 최종 검증 체크리스트

In [11]:
print("=" * 60)
print("최종 검증 체크리스트")
print("=" * 60)

checks = [
    ("Pickle 파일 생성", pickle_path.exists()),
    ("파일 크기 50MB 이하", file_size < 50),
    ("로드 시간 1초 이하", load_time < 1),
    ("168개 시간 패턴 존재", len(loaded_model['components']['time_patterns']) == 168),
    ("모델 버전 정보 포함", 'version' in loaded_model),
    ("카테고리 매핑 존재", len(loaded_model['components']['category_mapping']) > 0),
    ("Aisle 통계 존재", len(loaded_model['components']['aisle_stats']) > 0),
    ("글로벌 인기 Aisle 존재", len(loaded_model['components']['global_popular_aisles']) > 0),
]

all_passed = True
for check_name, passed in checks:
    status = "✓ PASS" if passed else "✗ FAIL"
    print(f"  [{status}] {check_name}")
    if not passed:
        all_passed = False

print("\n" + "=" * 60)
if all_passed:
    print("모든 검증 통과! Instacart Cold Start 모델 준비 완료.")
else:
    print("일부 검증 실패. 위 항목을 확인하세요.")
print("=" * 60)

최종 검증 체크리스트
  [✓ PASS] Pickle 파일 생성
  [✓ PASS] 파일 크기 50MB 이하
  [✓ PASS] 로드 시간 1초 이하
  [✓ PASS] 168개 시간 패턴 존재
  [✓ PASS] 모델 버전 정보 포함
  [✓ PASS] 카테고리 매핑 존재
  [✓ PASS] Aisle 통계 존재
  [✓ PASS] 글로벌 인기 Aisle 존재

모든 검증 통과! Instacart Cold Start 모델 준비 완료.


## 8. 완료

### 생성된 파일
- `pred/models/instacart_cold_start_v1.pkl` - Instacart Cold Start 모델

### 다음 단계
1. `InstacartColdStartModel`에서 이 Pickle 파일 로드하도록 수정
2. Phase 2: SelF Personalized SVD 모델 생성

### 사용 방법
```python
# pred/ml/models/instacart_cold_start.py에서
import pickle

with open('models/instacart_cold_start_v1.pkl', 'rb') as f:
    model_data = pickle.load(f)

time_patterns = model_data['components']['time_patterns']
category_mapping = model_data['components']['category_mapping']
```